In [13]:
import requests
from bs4 import BeautifulSoup
import re
import pandas as pd
import numpy as np
import time

restaurants_name = []
cost_for_two = []
ratings = []
cuisine_type = []
area = []
city_list = []

popular_cities = [
    "delhi-ncr", "mumbai", "bengaluru", "chennai", "pune", "kolkata",
    "goa", "ahmedabad", "jaipur", "agra", "hyderabad", "lucknow", "kochi",
    "coimbatore", "ranchi", "visakhapatnam", "patna", "amritsar", "bhubaneswar",
    "indore", "ludhiana", "mysuru", "vadodara", "thiruvananthapuram", "nagpur",
    "udaipur", "madurai", "chandigarh-tricity", "mangalore-tricity", "abu-dhabi", "dubai"
]

for city in popular_cities:
    link = "https://www.eazydiner.com"
    print(f"\nScraping: {city}")
    page = 1

    while True:
        url = f"{link}/restaurants?location={city}&page={page}"
        response = requests.get(url)
        soup = BeautifulSoup(response.text, "html.parser")

        restaurants = soup.find_all(
            "div",
            class_="flex align-v-start flex-between padding-t-15 padding-b-15 margin-t-12 relative full-width"
        )

        if not restaurants:
            break

        print(url)

        for rest in restaurants:
            #--------------Restaurant Name------------------#
            name_tag = rest.find("h3", class_="margin-0-auto all_unset")
            restaurants_name.append(name_tag.text.strip() if name_tag else "NA")

            #--------------Find all info blocks----------------------------#
            title_tags = rest.find_all(
                "div", class_="relative font-14 grey ellipsis listing_lh_18__xp_N9"
            )

            #--------------Area / Location------------------#
            location = title_tags[0].text.strip() if len(title_tags) > 0 else "NA"
            area.append(location)

            #--------------Cuisine Type------------------#
            if len(title_tags) > 1:
                potential_text = title_tags[1].text.strip()
                if "₹" in potential_text or "for" in potential_text.lower():
                    cuisine_type.append("NA")
                else:
                    cuisine_type.append(potential_text)
            else:
                cuisine_type.append("NA")

            #--------------Cost for Two (keep as text)------------------#
            price_text = None
            for tag in title_tags:
                text = tag.text.strip()
                if "₹" in text or "aed" in text.lower():
                    price_text = text
                    break

            # Keep the text as-is (no conversion)
            cost_for_two.append(price_text if price_text else "NA")

            #--------------Ratings (keep as text)------------------#
            rating_tag = rest.find("text")
            rating = rating_tag.text.strip() if rating_tag else "NA"
            ratings.append(rating)

            #--------------City------------------#
            city_list.append(city)

        page += 1
        time.sleep(1)

    print(f"Finished Scraping {city}: {page - 1}")




Scraping: delhi-ncr
https://www.eazydiner.com/restaurants?location=delhi-ncr&page=1
https://www.eazydiner.com/restaurants?location=delhi-ncr&page=2
https://www.eazydiner.com/restaurants?location=delhi-ncr&page=3
https://www.eazydiner.com/restaurants?location=delhi-ncr&page=4
https://www.eazydiner.com/restaurants?location=delhi-ncr&page=5
https://www.eazydiner.com/restaurants?location=delhi-ncr&page=6
https://www.eazydiner.com/restaurants?location=delhi-ncr&page=7
https://www.eazydiner.com/restaurants?location=delhi-ncr&page=8
https://www.eazydiner.com/restaurants?location=delhi-ncr&page=9
https://www.eazydiner.com/restaurants?location=delhi-ncr&page=10
https://www.eazydiner.com/restaurants?location=delhi-ncr&page=11
https://www.eazydiner.com/restaurants?location=delhi-ncr&page=12
https://www.eazydiner.com/restaurants?location=delhi-ncr&page=13
https://www.eazydiner.com/restaurants?location=delhi-ncr&page=14
https://www.eazydiner.com/restaurants?location=delhi-ncr&page=15
https://www.e

### Creating a DataFrame

In [14]:
df = pd.DataFrame({
    "Restaurant Name": restaurants_name,
    "City": city_list,
    "Area": area,
    "Cuisine Type": cuisine_type,
    "Cost for Two": cost_for_two,
    "Ratings": ratings
})
df

,Restaurant Name,City,Area,Cuisine Type,Cost for Two,Ratings
0,Desi Villagio,delhi-ncr,"Connaught Place (CP), Central Delhi",Indian,₹1300 for two,4.1
1,Hard Rock Cafe,delhi-ncr,"Connaught Place (CP), Central Delhi",Multicuisine,₹1500 for two,4.0
2,Cafe Out of the Box Courtyard,delhi-ncr,"Connaught Place (CP), Central Delhi",Multicuisine,₹1200 for two,3.9
3,Sakura,delhi-ncr,"The Metropolitan Hotel & Spa, New Delhi","Japanese, Sushi",₹4500 for two,4.0
4,Shang Palace,delhi-ncr,Shangri-La Eros New Delhi,Chinese,₹3000 for two,4.5
...,...,...,...,...,...,...
17017,New Silver Paradise,dubai,Rove Downtown Dubai,Indian,AED100 for two,4.0
17018,Anaarkali Restaurant,dubai,"Al Karama, Dubai",Multicuisine,AED100 for two,4.0
17019,Zaam’s Restaurant,dubai,"Al Hamriya, Dubai, Dubai",Indian,AED70 for two,4.0
17020,Layali Aldhan Restaurant,dubai,"Al Barsha, Dubai","Egyptian, Arabian",AED120 for two,4.0


In [16]:
df.to_csv("Eazy_Diner.csv", index = False)